In [7]:
# %% [markdown]
# # 📊 Relatórios Gráficos de RMSE por Método de Imputação
# 
# Este notebook carrega um `results.csv` com colunas:
# `file, rate, method, rmse, nrmse, nrmse_mean, mae, mape, r2`  
# e produz:
# 
# 1) **Gráficos por (file, rate)** — um gráfico de linha por combinação (dataset X missing rate Y), comparando RMSE entre métodos.  
# 2) **Comparações gerais** — linhas com médias por **rate** (global) e linhas por **file** (eixo = rate, uma linha por método).  
# 3) **Relatório final** — médias de RMSE por método, vencedores por dataset e “quão melhor” foi o melhor método.
# 
# > **Configuração rápida:** informe o caminho do arquivo e a métrica principal (default = `rmse`).

# %%
# =============================
# CONFIGURAÇÕES INICIAIS
# =============================
RESULTS_FILE = 'results_total.csv'   # <- ajuste aqui se necessário
METRIC = 'rmse'                # Opções típicas: 'rmse' (default)

import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Carrega o CSV
df = pd.read_csv(RESULTS_FILE)

# Checagens básicas
required_cols = {'file','rate','method', METRIC}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"CSV faltando colunas: {missing_cols}")

# Tipos/correções
df['file'] = pd.to_numeric(df['file'], errors='coerce').astype('Int64')
df['rate'] = pd.to_numeric(df['rate'], errors='coerce').astype('Int64')
df = df.dropna(subset=['file','rate', 'method', METRIC]).copy()
df['method'] = df['method'].astype(str).str.strip()

# Normaliza nomes opcionais (ex.: aspas ao redor do método)
df['method'] = df['method'].str.replace('"','').str.replace("'", '')

# Ordenações úteis
method_order = sorted(df['method'].unique().tolist())
rate_order   = [10,20,30,40]
file_order   = sorted(df['file'].unique().tolist())

# Paleta de cores por método (fixa e consistente ao longo dos gráficos)
# (pedido do usuário para cores distintas por método)
cmap = plt.cm.get_cmap('tab20', len(method_order))
method_to_color = {m: cmap(i) for i, m in enumerate(method_order)}

print(f"Carregado: {len(df)} linhas | files={len(file_order)} | rates encontrados={sorted(df['rate'].unique())}")
print(f"Métricas disponíveis: {set(df.columns) & {'rmse','nrmse','nrmse_mean','mae','mape','r2'}}")

# # %% [markdown]
# # ## 1) Gráficos por (file, rate)
# # Para cada dataset (`file`) e para cada `rate` (10, 20, 30, 40), plota-se um gráfico de linha simples
# # comparando o **RMSE** de todos os métodos.  
# # 
# # **Observações de estilo (exigências):**
# # - Gráficos **sem subplots** (cada figura é independente).
# # - Usar **matplotlib** (sem seaborn).
# # - **Cores por método** (paleta consistente).
# # - Eixo X categórico (métodos), com pontos marcados; a linha apenas conecta visualmente os métodos.

# # %%
# # ==============
# # PLOTS (file, rate)
# # ==============
# def plot_file_rate_group(gdf, file_id, rate):
#     gdf = gdf.copy()
#     # garante a ordem de métodos
#     gdf['method'] = pd.Categorical(gdf['method'], categories=method_order, ordered=True)
#     gdf = gdf.sort_values('method')

#     plt.figure(figsize=(10, 5))
#     # Desenho: uma linha conectando os pontos (para cumprir "gráficos de linha")
#     # e cada ponto colorido pelo método (cores distintas por método), com legenda.
#     x = np.arange(len(gdf))
#     y = gdf[METRIC].to_numpy()
#     plt.plot(x, y, marker='o')  # uma linha conectando os métodos

#     # pinta cada ponto de acordo com o método e cria entrada de legenda
#     for xi, yi, m in zip(x, y, gdf['method']):
#         plt.scatter([xi], [yi], s=60, color=method_to_color[str(m)], label=str(m))

#     # Para evitar entradas repetidas na legenda, consolidamos com dict
#     handles, labels = plt.gca().get_legend_handles_labels()
#     seen = set()
#     new_handles, new_labels = [], []
#     for h, l in zip(handles, labels):
#         if l not in seen:
#             new_handles.append(h); new_labels.append(l); seen.add(l)

#     plt.legend(new_handles, new_labels, title='Método', bbox_to_anchor=(1.02, 1), loc='upper left')
#     plt.title(f'RMSE por Método — file={file_id}, rate={rate}%')
#     plt.ylabel(METRIC.upper())
#     plt.xlabel('Método de Imputação')
#     plt.xticks(x, gdf['method'], rotation=30, ha='right')
#     plt.grid(True, linestyle='--', alpha=0.3)
#     plt.tight_layout()
#     plt.show()

# # Itera em ordem definida (file, rate)
# for f in file_order:
#     for r in rate_order:
#         subset = df[(df['file'] == f) & (df['rate'] == r)]
#         if len(subset) == 0:
#             continue
#         plot_file_rate_group(subset, f, r)

# # %% [markdown]
# # ## 2) Comparações Gerais
# # Produzimos dois conjuntos de comparações em **gráficos de linha**:
# # 
# # **(A) Global por rate (média em todos os files):**  
# # Eixo X = `rate`, **uma linha por método**, valor = média de `RMSE`.
# # 
# # **(B) Por file (um gráfico por file):**  
# # Eixo X = `rate`, **uma linha por método**, valor = `RMSE` de cada (file, rate, method).

# # %%
# # ==============
# # (A) GLOBAL por rate (média entre files)
# # ==============
# global_mean = (
#     df.groupby(['rate','method'], as_index=False)[METRIC]
#       .mean()
# )

# plt.figure(figsize=(10,5))
# for m in method_order:
#     mdf = global_mean[global_mean['method'] == m].copy()
#     mdf = mdf.sort_values('rate')
#     if len(mdf):
#         plt.plot(mdf['rate'], mdf[METRIC], marker='o', label=m, color=method_to_color[m])

# plt.title('MÉDIA GLOBAL de RMSE por RATE (linha = método)')
# plt.xlabel('Missing Rate (%)')
# plt.ylabel(METRIC.upper())
# plt.grid(True, linestyle='--', alpha=0.3)
# plt.legend(title='Método', bbox_to_anchor=(1.02, 1), loc='upper left')
# plt.tight_layout()
# plt.show()

# # ==============
# # (B) Por file: linhas por método (x = rate)
# # ==============
# def plot_file_lines(gdf, file_id):
#     plt.figure(figsize=(10,5))
#     for m in method_order:
#         mdf = gdf[gdf['method'] == m].copy()
#         if len(mdf) == 0:
#             continue
#         mdf = mdf.sort_values('rate')
#         plt.plot(mdf['rate'], mdf[METRIC], marker='o', label=m, color=method_to_color[m])
#     plt.title(f'RMSE por RATE — file={file_id} (linha = método)')
#     plt.xlabel('Missing Rate (%)')
#     plt.ylabel(METRIC.upper())
#     plt.grid(True, linestyle='--', alpha=0.3)
#     plt.legend(title='Método', bbox_to_anchor=(1.02, 1), loc='upper left')
#     plt.tight_layout()
#     plt.show()

# for f in file_order:
#     g = df[df['file'] == f]
#     if len(g):
#         plot_file_lines(g, f)

# %% [markdown]
# ## 3) Relatório Final (texto + tabelas)
# - **Média de RMSE por método (global).**  
# - **Vencedor por dataset (`file`)** (considerando o `rate` médio do dataset, ou o melhor método pelo *RMSE médio* dentro do file).  
# - **Vantagem do vencedor** — quanto melhor (%) o melhor método foi em relação ao **2º melhor** no mesmo dataset.
# 
# > Obs.: para o “vencedor por file”, agregamos o RMSE por método dentro do `file` tirando a **média** sobre os rates disponíveis.

# %%
# =============================
# RELATÓRIO FINAL
# =============================

# 1) Média global por método
mean_by_method = (
    df.groupby('method', as_index=False)[METRIC]
      .mean()
      .sort_values(METRIC)
      .reset_index(drop=True)
)

# 2) Vencedor por file (média dos rates por método dentro do file)
file_method_mean = (
    df.groupby(['file','method'], as_index=False)[METRIC]
      .mean()
)

winners = []
for f in file_order:
    sub = file_method_mean[file_method_mean['file'] == f].copy()
    if len(sub) == 0:
        continue
    sub = sub.sort_values(METRIC)
    best = sub.iloc[0]
    if len(sub) >= 2:
        second = sub.iloc[1]
        # melhoria relativa do 1º para o 2º (quanto menor o RMSE, melhor)
        improvement = (second[METRIC] - best[METRIC]) / second[METRIC] * 100.0
    else:
        improvement = np.nan
    winners.append({
        'file': f,
        'best_method': best['method'],
        f'mean_{METRIC}_best': best[METRIC],
        'second_best_method': second['method'] if len(sub) >= 2 else None,
        f'mean_{METRIC}_second_best': second[METRIC] if len(sub) >= 2 else None,
        'improvement_vs_second_best_%': improvement
    })

winners_df = pd.DataFrame(winners).sort_values('file').reset_index(drop=True)

# 3) Tabela resumida: quantas vezes cada método venceu
win_counts = winners_df['best_method'].value_counts().rename_axis('method').reset_index(name='wins')
win_counts = win_counts.sort_values('wins', ascending=False).reset_index(drop=True)

# =============================
# RELATÓRIO FINAL
# =============================

# 1) Média global por método
mean_by_method = (
    df.groupby('method', as_index=False)[METRIC]
      .mean()
      .sort_values(METRIC)
      .reset_index(drop=True)
)

# 2) Vencedor por file (média dos rates por método dentro do file)
file_method_mean = (
    df.groupby(['file','method'], as_index=False)[METRIC]
      .mean()
)

winners = []
for f in file_order:
    sub = file_method_mean[file_method_mean['file'] == f].copy()
    if len(sub) == 0:
        continue
    sub = sub.sort_values(METRIC)
    best = sub.iloc[0]
    if len(sub) >= 2:
        second = sub.iloc[1]
        improvement = (second[METRIC] - best[METRIC]) / second[METRIC] * 100.0
    else:
        improvement = np.nan
    winners.append({
        'file': f,
        'best_method': best['method'],
        f'mean_{METRIC}_best': best[METRIC],
        'second_best_method': second['method'] if len(sub) >= 2 else None,
        f'mean_{METRIC}_second_best': second[METRIC] if len(sub) >= 2 else None,
        'improvement_vs_second_best_%': improvement
    })

winners_df = pd.DataFrame(winners).sort_values('file').reset_index(drop=True)

# 3) Tabela resumida: quantas vezes cada método venceu
win_counts = winners_df['best_method'].value_counts().rename_axis('method').reset_index(name='wins')
win_counts = win_counts.sort_values('wins', ascending=False).reset_index(drop=True)

# ========== PRINT/EXIBIÇÃO ==========
print("=== Média GLOBAL de RMSE por método ===")
print(mean_by_method.to_string(index=False))

print("\n=== Vencedor por dataset (file) ===")
print(winners_df.to_string(index=False))

print("\n=== Contagem de vitórias por método ===")
print(win_counts.to_string(index=False))

# Texto resumido: lista dos melhores por file
print("\n=== Resumo textual por file ===")
for _, row in winners_df.iterrows():
    f = row['file']
    bm = row['best_method']
    best_val = row[f'mean_{METRIC}_best']
    second = row['second_best_method']
    second_val = row[f'mean_{METRIC}_second_best']
    imp = row['improvement_vs_second_best_%']
    if pd.notna(second):
        print(f"- file {f}: melhor = {bm} (média {METRIC}={best_val:.4f}); "
              f"2º={second} ({second_val:.4f}); melhoria ≈ {imp:.2f}%")
    else:
        print(f"- file {f}: melhor = {bm} (média {METRIC}={best_val:.4f}); sem 2º melhor disponível.")


Carregado: 10089 linhas | files=282 | rates encontrados=[np.int64(10), np.int64(20), np.int64(30), np.int64(40)]
Métricas disponíveis: {'nrmse', 'mape', 'nrmse_mean', 'mae', 'r2', 'rmse'}
=== Média GLOBAL de RMSE por método ===
                   method         rmse
     linear interpolation 2.226074e+06
     kalman arima (1,1,1) 2.649497e+06
  Temporal_Matrix+SVD+KNN 2.686930e+06
kalman structural (level) 2.832325e+06
                    ffill 2.928377e+06
         ewma (alpha=0.2) 3.046291e+06
        knn imputer (k=5) 3.678978e+06
                     mean 3.678978e+06
              soft_impute 6.205311e+06

=== Vencedor por dataset (file) ===
 file               best_method  mean_rmse_best        second_best_method  mean_rmse_second_best  improvement_vs_second_best_%
    0      linear interpolation    2.389616e+07   Temporal_Matrix+SVD+KNN           4.105770e+07                     41.798582
    1      linear interpolation    2.802467e+07      kalman arima (1,1,1)           4.37122

/tmp/ipykernel_3293597/2361575642.py:52: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', len(method_order))
